In [ ]:
import tensorflow_hub as hub
import tensorflow as tf
import numpy as np
import os

In [ ]:
def generate_random_images(model, num_images=1):
    images = model(tf.random.normal([num_images, 512]))['default'] 
    return images.numpy() 

def interpolate_vectors(v1: np.ndarray, v2: np.ndarray, steps: int = 11) -> np.ndarray:
    lambdas = np.linspace(0, 1, steps)
    return np.array([(1 - l) * v1 + l * v2 for l in lambdas])

def generate_morphed_images(model, vectors: np.ndarray) -> list:
    latent_batch = tf.convert_to_tensor(vectors, dtype=tf.float32)
    return model(latent_batch)['default']
    
def show_images(images):
    import matplotlib.pyplot as plt
    import math

    n = len(images)
    if n <= 4:
        rows, cols = 1, n
    else:
        rows = 2
        cols = math.ceil(n / 2)

    plt.figure(figsize=(cols * 2.5, rows * 2.5))  

    for i in range(n):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(images[i])
        plt.axis('off')
    plt.tight_layout()
    plt.show()


# Load GAN

In [ ]:
os.environ['TFHUB_CACHE_DIR'] = './tfhub_cache'  # if used on deep server, since it is not permitted to write to /tmp 
gan = hub.load("https://tfhub.dev/google/progan-128/1").signatures['default'] 

# Generate random images

In [ ]:
images = generate_random_images(gan, num_images=4)
show_images(images) 

# Morphing in latent space

In [ ]:
random_vector1, random_vector2 = tf.random.normal([512]), tf.random.normal([512])
interpolated_vectors = interpolate_vectors(random_vector1, random_vector2)
morphed_images = generate_morphed_images(gan, interpolated_vectors)
show_images(morphed_images)